# 第12回：深層学習を体験する

8×8画素の手書き数字を，隠れ層が二つある全結合ネットワーク（MLP）で分類します．学習曲線，検証によるモデル選択，学習済みモデルでの推論，誤分類を観察します．

CPUで実行できる小さな教材です．GPUや有料サービスは不要です．scikit-learnに含まれるデータを使い，外部画像のダウンロードはしません．

**条件の変更は検証データで比較し，テストは設定を決めた後の最終確認に使います．**

In [ ]:
# Google Colabでは通常，このライブラリが用意されています．
# ローカル環境で不足している場合だけ，必要なライブラリを入れます．
import importlib.util, subprocess, sys
packages = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib', 'sklearn':'scikit-learn'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
print('Python:', sys.version.split()[0], 'scikit-learn:', sklearn.__version__)


## 1．設定とデータ

最初はそのまま実行します．比較実験では `LEARNING_RATE` または `MAX_EPOCHS` の一つを変えます．ランダムシード，分割，その他の条件はそろえます．

In [ ]:
SEED = 42
HIDDEN_LAYERS = (32, 16)
LEARNING_RATE = .001
BATCH_SIZE = 64
MAX_EPOCHS = 30
ALPHA = .0001
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss
from copy import deepcopy
digits = load_digits()
images = digits.images
y = digits.target
# 既知の画素範囲0〜16を使う固定の変換．データ全体の統計量は使わない．
X = digits.data.astype(np.float64) / 16.0
print('画像:', images.shape, '入力:', X.shape, '画素範囲:', images.min(), images.max())
print('クラスごとの件数:', np.bincount(y))
fig, axes = plt.subplots(2,5,figsize=(9,4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i], cmap='gray', vmin=0, vmax=16); ax.set_title(f'label={y[i]}'); ax.axis('off')
plt.tight_layout(); plt.show()


## 2．データを分割する

テスト20％を取り分け，残りの25％を検証にします．今回のデータでは訓練1,077件，検証360件，テスト360件です．画像を横一列に並べた64個の数値を入力にします．

In [ ]:
all_idx = np.arange(len(y))
trainval_idx, test_idx = train_test_split(all_idx, test_size=.2, stratify=y, random_state=SEED)
train_idx, val_idx = train_test_split(trainval_idx, test_size=.25, stratify=y[trainval_idx], random_state=SEED)
assert not (set(train_idx)&set(val_idx) or set(train_idx)&set(test_idx) or set(val_idx)&set(test_idx))
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
print('訓練・検証・テスト:', len(train_idx), len(val_idx), len(test_idx))


## 3．ネットワークを作る

入力64 → 隠れ32 → 隠れ16 → 出力10．隠れ層ではReLUを使います．学習時に重みとバイアスを更新し，推論時にはその値を使います．

今回は全結合のMLPを使います．CNNの局所的な計算や重みの共有は第7回の発展内容です．

In [ ]:
model = MLPClassifier(hidden_layer_sizes=HIDDEN_LAYERS, activation='relu', solver='adam',
                      learning_rate_init=LEARNING_RATE, batch_size=BATCH_SIZE, alpha=ALPHA,
                      random_state=SEED, shuffle=True)
history = []
best_model = None
best_val_loss = float('inf')
best_epoch = None


## 4．1エポックずつ学習して観察する

`partial_fit` を一度呼ぶと，訓練データを1回学習します．検証データでは推論するだけで，重みの更新には使いません．検証の交差エントロピーが最も小さい時点のコピーを保存します．

下の訓練損失 `loss_` はミニバッチで計算した正則化込みの学習中の値です．検証損失はエポック終了時の正則化なしの交差エントロピーです．同じ定義の厳密な差ではなく，それぞれの推移を読みます．

In [ ]:
for epoch in range(1, MAX_EPOCHS+1):
    model.partial_fit(X_train, y_train, classes=np.arange(10))
    val_proba = model.predict_proba(X_val)
    val_loss = log_loss(y_val, val_proba, labels=np.arange(10))
    row = {'epoch':epoch, 'training_loss':float(model.loss_), 'validation_loss':float(val_loss),
           'train_accuracy':accuracy_score(y_train, model.predict(X_train)),
           'validation_accuracy':accuracy_score(y_val, model.predict(X_val))}
    history.append(row)
    if val_loss < best_val_loss:
        best_val_loss, best_epoch, best_model = val_loss, epoch, deepcopy(model)
    if epoch==1 or epoch%5==0:
        print(f'epoch={epoch:2d} train_loss={model.loss_:.3f} val_loss={val_loss:.3f} val_acc={row["validation_accuracy"]:.3f}')
history_df = pd.DataFrame(history)
print('検証損失で選んだエポック:', best_epoch)
print('重みの形:', [w.shape for w in best_model.coefs_])
print('学ぶ値の数:', sum(w.size for w in best_model.coefs_)+sum(b.size for b in best_model.intercepts_))


## 5．学習曲線を読む

損失は小さい方がよい量，正解率は大きい方がよい量です．検証損失が下がり続けるとは限りません．訓練損失だけを見て学習回数を決めないようにします．

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].plot(history_df.epoch, history_df.training_loss, label='training loss (includes regularization)')
axes[0].plot(history_df.epoch, history_df.validation_loss, label='validation cross-entropy')
axes[0].axvline(best_epoch, color='gray', linestyle='--'); axes[0].set(xlabel='epoch', ylabel='loss'); axes[0].legend(fontsize=8)
axes[1].plot(history_df.epoch, history_df.train_accuracy, label='train')
axes[1].plot(history_df.epoch, history_df.validation_accuracy, label='validation')
axes[1].set(xlabel='epoch', ylabel='accuracy', ylim=(0,1)); axes[1].legend()
plt.tight_layout(); plt.show()


## 6．学習済みモデルを使う

コピーした `best_model` で検証画像の推論を行います．`predict` や `predict_proba` は重みを更新しません．確率が高い出力も，正しいことを保証しません．

In [ ]:
sample = 0
proba = best_model.predict_proba(X_val[sample:sample+1])[0]
print('正解:', int(y_val[sample]), '予測:', int(proba.argmax()), '最大の出力確率:', round(float(proba.max()),3))
print('各クラスの出力:', np.round(proba,3))
plt.imshow(images[val_idx[sample]], cmap='gray', vmin=0, vmax=16); plt.axis('off'); plt.show()


## 7．検証データの誤分類を調べる

正解と予測の組合せ，画像の形，誤分類の数を見ます．人が見ても紛らわしいものがあるか考えます．誤分類が0件なら，その事実とデータ数を記録します．

In [ ]:
val_pred = best_model.predict(X_val)
ConfusionMatrixDisplay.from_predictions(y_val, val_pred, labels=np.arange(10))
plt.show()
wrong = np.flatnonzero(val_pred != y_val)
print('検証の誤分類件数:', len(wrong))
if len(wrong):
    fig, axes = plt.subplots(3,3,figsize=(7,7))
    for ax in axes.flat: ax.axis('off')
    for ax, k in zip(axes.flat, wrong[:9]):
        ax.imshow(images[val_idx[k]], cmap='gray', vmin=0, vmax=16)
        ax.set_title(f'true={y_val[k]}, pred={val_pred[k]}')
    plt.tight_layout(); plt.show()
else:
    print('この検証データでは誤分類0件．他の筆跡でも正しいとは限りません．')


## 8．比較実験（ここで一度止まる）

- 学習率を0.001から0.01へ変え，**検証**の学習曲線を比較する．または学習回数だけを変える．
- 他の条件とデータ分割は同じにし，最良エポック・検証損失・正解率を記録する．
- 条件を決めて，最後のテストへ進む．

授業ではセルごとに実行します．全セルの一括実行は，初期設定でテストまで進みます．

## 9．最終テスト

検証で選んだ `best_model` を取り分けた360件に適用します．この教材では，選んだモデルをそのまま評価し，訓練＋検証での再学習は行いません．テストを見て再調整すると独立した評価ではなくなります．

In [ ]:
test_pred = best_model.predict(X[test_idx])
test_accuracy = accuracy_score(y[test_idx], test_pred)
print('最終テスト正解率:', round(test_accuracy,4))
print('テスト件数:',len(test_idx), '正解件数:',int((test_pred==y[test_idx]).sum()))
ConfusionMatrixDisplay.from_predictions(y[test_idx], test_pred, labels=np.arange(10))
plt.show()
print({'seed':SEED, 'hidden_layers':HIDDEN_LAYERS, 'learning_rate':LEARNING_RATE,
       'batch_size':BATCH_SIZE, 'max_epochs':MAX_EPOCHS, 'alpha':ALPHA, 'selected_epoch':best_epoch,
       'validation_loss':float(best_val_loss), 'test_accuracy':float(test_accuracy),
       'sklearn_version':sklearn.__version__})


## 10．提出する短い記録

設定，選んだエポック，検証・テストの件数と指標，学習曲線の読み方，誤分類の例，変更した条件の影響をまとめます．

この小さなデータで高い正解率が出ても，他の筆跡・撮影環境・大きな画像への有効性は別の評価が必要です．環境の版によって数値は多少変わります．

## 参考資料

- [Digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)
- [Neural network models](https://scikit-learn.org/stable/modules/neural_networks_supervised.html)
- [MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)

説明とコードは授業用に作成したものです．データセットの由来はscikit-learnの資料を参照してください．